<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

... (80 outputs left out)

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace fo

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace fo

Run date time (IST): 2026-11-02 14:41:45
=================== PORTFOLIO QUALITY ====================
qualified stocks: 86
with latest results: 86
still star stocks: 46
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.47 C
CAGR/XIRR %:     -0.62%
Est FTT Value:   2.74 C
Future Upside:   1.28 C (86.83%)
=================== PERFORMANCE ==========================
Current PnL:             -49.74 L (-25.92%)
Total Booked PnL:        48.59 L (32.6%)
Booked + Current PnL:    -1.16 L (-0.78%)
CY Booked PnL:           5.71 L (3.89%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               1.59 L (1.1%)
CY Investment:            1.92 C
Reserve:                  1.90 K
=================== P&L DETAIL ============================
Total Profit:             1.52 L
Total Loss:               -51.26 L


/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:604: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace 

In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
85,ZYDUSLIFE,-0.01,31.33,3.18,35.51,8752.0,65659.0,275233.0,1285.01,29.99,71.0,H-LC,11.29,120.0,7.50,1.87,44.84,AR,ATH,PHARMA
76,TTKPRESTIG,2.81,-31.71,46.57,0.09,32050.0,-31956.0,68821.0,770.00,-16.95,49.0,M-SC,5.13,250.0,-1.00,0.47,23.87,OX40N,NTT,DURABLES
12,BOSCH-HCIL,2.50,-25.89,34.92,-0.01,33076.0,-33087.0,94718.0,2282.00,738.08,43.0,M-SC,22.88,231.0,-1.00,0.65,61.22,OX40N,BTT,AC
40,ICICIPRULI,-0.47,-32.15,27.40,-13.56,36623.0,-63345.0,133660.0,522.35,-56.46,25.0,X-MC,8.49,78.0,-1.73,0.91,0.11,X40,ATH,INSURANCE
66,SIS,3.18,-12.51,38.60,21.26,37497.0,-13890.0,97142.0,528.00,86.48,42.0,H-SC,8.86,168.0,-0.37,0.66,44.44,OX40N,NTT,MISC


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
52,MARUTI,1.75,-15.27,47.62,25.08,121495.0,-45984.0,255134.0,17120.06,-20.65,49.0,X-LC,14.77,33.0,-0.38,1.74,3.82,XR,ATH,AUTO
38,HYUNDAI,3.67,2.47,40.46,43.94,104202.0,6211.0,257543.0,2782.73,6.11,51.0,X-LC,6.85,29.0,0.06,1.75,18.25,X40N,ATH,AUTO
51,M&M,0.98,-10.45,31.20,17.48,84116.0,-31474.0,269601.0,3762.88,-21.93,44.0,X-LC,10.35,25.0,-0.37,1.84,4.25,X40N,ATH,AUTO
4,ASIANPAINT,0.31,-15.63,47.40,24.36,132377.0,-51756.0,279276.0,3430.49,-26.14,44.0,X-LC,7.05,30.0,-0.39,1.90,10.66,X40,ATH,PAINTS
23,DMART,0.49,-21.04,67.72,32.43,189393.0,-74530.0,279670.0,5391.45,-34.26,34.0,X-LC,15.59,14.0,-0.39,1.90,2.58,X40N,ATH,FMCG


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
22,DIXON,12.21,-10.08,53.26,37.82,124904.0,-26281.0,234517.0,18917.17,-65.76,51.0,X-MC,9.23,54.0,-0.21,1.60,27.70,X40N,ATH,IT
61,RELIANCE,8.48,-11.63,54.47,36.51,172084.0,-41562.0,315925.0,1952.00,-10.27,61.0,X-LC,10.10,21.0,-0.24,2.15,9.13,XY25,BTT,REFINERIES
73,TITAGARH,7.10,-34.04,120.95,45.74,193202.0,-82440.0,159737.0,1548.00,NaN,41.0,NaN,8.23,175.0,-0.43,1.09,22.02,XY24,NTT,ENGINEERING
41,IEX,6.33,-27.19,104.12,48.63,151591.0,-54361.0,145593.0,219.00,NaN,51.0,NaN,13.47,137.0,-0.36,0.99,6.33,XR,NTT,MISC
60,RELAXO,5.10,-62.91,310.61,52.31,167242.0,-91317.0,53843.0,1176.00,NaN,46.0,NaN,14.43,92.0,-0.55,0.37,21.74,X40N,NTT,FOOTWEAR


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
43,INDIGOPNTS,-12.41,-37.95,61.19,0.02,66277.0,-66245.0,108314.0,1408.00,-40.91,20.0,M-SC,8.16,222.0,-1.00,0.74,23.41,OX40N,NTT,PAINTS
5,ATULAUTO,-4.93,-33.37,118.48,45.57,155614.0,-65790.0,131342.0,844.00,NaN,35.0,NaN,10.22,248.0,-0.42,0.89,1.75,XY24,NTT,AUTO
67,STARHEALTH,-2.73,-0.17,47.51,47.26,133337.0,-473.0,280650.0,761.00,-11.87,37.0,H-SC,5.79,173.0,-0.00,1.91,21.35,XY24,NTT,INSURANCE
45,INFY,-2.62,-16.51,70.88,42.66,237207.0,-66195.0,334660.0,1972.00,-39.69,58.0,X-LC,9.43,3.0,-0.28,2.28,17.12,X40,NTT,IT
39,ICICIGI,-2.39,-18.27,44.83,18.38,73323.0,-36551.0,163558.0,2234.79,-18.45,41.0,X-MC,9.38,66.0,-0.50,1.11,8.54,X40,ATH,INSURANCE


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
67,STARHEALTH,-2.73,-0.17,47.51,47.26,133337.0,-473.0,280650.0,761.0,-11.87,37.0,H-SC,5.79,173.0,-0.0,1.91,21.35,XY24,NTT,INSURANCE


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.13,44.20,43.17,106.44,91909.0,65256.0,212900.0,1629.94,5.26,35.0,M-SC,26.73,220.0,0.71,1.45,61.31,OX40N,ATH,CABLES
66,SIS,3.18,-12.51,38.60,21.26,37497.0,-13890.0,97142.0,528.00,86.48,42.0,H-SC,8.86,168.0,-0.37,0.66,44.44,OX40N,NTT,MISC
12,BOSCH-HCIL,2.50,-25.89,34.92,-0.01,33076.0,-33087.0,94718.0,2282.00,738.08,43.0,M-SC,22.88,231.0,-1.00,0.65,61.22,OX40N,BTT,AC
76,TTKPRESTIG,2.81,-31.71,46.57,0.09,32050.0,-31956.0,68821.0,770.00,-16.95,49.0,M-SC,5.13,250.0,-1.00,0.47,23.87,OX40N,NTT,DURABLES
43,INDIGOPNTS,-12.41,-37.95,61.19,0.02,66277.0,-66245.0,108314.0,1408.00,-40.91,20.0,M-SC,8.16,222.0,-1.00,0.74,23.41,OX40N,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.13,44.20,43.17,106.44,91909.0,65256.0,212900.0,1629.94,5.26,35.0,M-SC,26.73,220.0,0.71,1.45,61.31,OX40N,ATH,CABLES
85,ZYDUSLIFE,-0.01,31.33,3.18,35.51,8752.0,65659.0,275233.0,1285.01,29.99,71.0,H-LC,11.29,120.0,7.50,1.87,44.84,AR,ATH,PHARMA


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.13,44.2,43.17,106.44,91909.0,65256.0,212900.0,1629.94,5.26,35.0,M-SC,26.73,220.0,0.71,1.45,61.31,OX40N,ATH,CABLES


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.13,44.20,43.17,106.44,91909.0,65256.0,212900.0,1629.94,5.26,35.0,M-SC,26.73,220.0,0.71,1.45,61.31,OX40N,ATH,CABLES
84,WIPRO,-0.83,-9.62,113.56,93.01,256839.0,-24070.0,226170.0,420.00,-23.61,71.0,M-LC,9.01,98.0,-0.09,1.54,25.44,XR,NTT,IT
80,VALIANTORG,4.60,-9.09,91.56,74.15,136180.0,-14872.0,148733.0,838.00,NaN,52.0,NaN,51.07,145.0,-0.11,1.01,121.34,XR,NTT,CHEMICALS
70,TANLA,3.67,-27.22,195.66,115.17,385546.0,-73712.0,197049.0,1903.93,NaN,77.0,NaN,8.15,147.0,-0.19,1.34,74.19,AR,ATH,IT
3,ALKYLAMINE,2.78,-30.40,176.66,92.54,124132.0,-30697.0,70266.0,4520.85,NaN,37.0,NaN,10.22,142.0,-0.25,0.48,33.43,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
37,HONAUT,-1.07,-24.35,99.46,50.88,145087.0,-46963.0,145875.0,58192.79,-40.89,22.0,X-SC,5.68,90.0,-0.32,0.99,11.14,X40N,ATH,ELECTRICAL
40,ICICIPRULI,-0.47,-32.15,27.40,-13.56,36623.0,-63345.0,133660.0,522.35,-56.46,25.0,X-MC,8.49,78.0,-1.73,0.91,0.11,X40,ATH,INSURANCE
19,COLPAL,-0.16,-40.39,131.09,37.76,205811.0,-106365.0,157000.0,3628.04,-27.76,33.0,X-MC,13.89,62.0,-0.52,1.07,0.00,XY25,ATH,FMCG
6,AWL,1.08,-42.85,192.26,67.02,391482.0,-152688.0,203621.0,485.00,-57.79,34.0,X-SC,6.57,81.0,-0.39,1.39,1.23,XY24,NTT,FMCG
23,DMART,0.49,-21.04,67.72,32.43,189393.0,-74530.0,279670.0,5391.45,-34.26,34.0,X-LC,15.59,14.0,-0.39,1.90,2.58,X40N,ATH,FMCG


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PAGEIND,2.19,-8.05,44.48,32.85,109770.0,-21594.0,246785.0,50936.08,-31.61,48.0,X-MC,1.04,55.0,-0.20,1.68,17.84,X40,ATH,APPARELS
1,ABBOTINDIA,0.78,-13.33,33.11,15.37,51324.0,-23845.0,155010.0,34390.26,-29.45,48.0,X-MC,1.59,58.0,-0.46,1.06,5.00,X40,ATH,PHARMA
17,CIPLA,-0.14,3.51,19.69,23.89,61276.0,10547.0,311202.0,1648.14,17.48,58.0,X-LC,2.50,18.0,0.17,2.12,16.56,X40N,ATH,PHARMA
11,BERGEPAINT,0.67,-13.72,45.24,25.32,100611.0,-35354.0,222395.0,680.00,-24.95,50.0,X-MC,2.57,68.0,-0.35,1.51,16.62,XY24,NTT,PAINTS
33,HCLTECH,0.38,-1.91,28.15,25.70,99672.0,-6899.0,354074.0,1837.07,-6.73,71.0,X-LC,3.21,8.0,-0.07,2.41,41.27,X40,ATH,IT


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
28,GODREJCP,-0.43,-25.91,79.37,32.90,146749.0,-64645.0,184892.0,1460.97,-27.24,34.0,X-MC,16.95,32.0,-0.44,1.26,0.00,X40N,ATH,FMCG
19,COLPAL,-0.16,-40.39,131.09,37.76,205811.0,-106365.0,157000.0,3628.04,-27.76,33.0,X-MC,13.89,62.0,-0.52,1.07,0.00,XY25,ATH,FMCG
40,ICICIPRULI,-0.47,-32.15,27.40,-13.56,36623.0,-63345.0,133660.0,522.35,-56.46,25.0,X-MC,8.49,78.0,-1.73,0.91,0.11,X40,ATH,INSURANCE
6,AWL,1.08,-42.85,192.26,67.02,391482.0,-152688.0,203621.0,485.00,-57.79,34.0,X-SC,6.57,81.0,-0.39,1.39,1.23,XY24,NTT,FMCG
35,HDFCLIFE,-0.93,-9.92,56.04,40.56,157069.0,-30879.0,280280.0,811.41,-34.42,42.0,X-LC,10.39,31.0,-0.20,1.91,1.25,X40,ATH,INSURANCE


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,0.28,-48.52,154.50,31.02,169826.0,-103600.0,109920.0,17484.41,-62.16,41.0,X-SC,25.29,57.0,-0.61,0.75,3.48,X40,ATH,FMCG
40,ICICIPRULI,-0.47,-32.15,27.40,-13.56,36623.0,-63345.0,133660.0,522.35,-56.46,25.0,X-MC,8.49,78.0,-1.73,0.91,0.11,X40,ATH,INSURANCE
2,ACC,1.74,-48.55,239.74,74.81,330755.0,-130164.0,137964.0,3906.00,-49.01,43.0,X-SC,16.63,85.0,-0.39,0.94,2.20,XY24,BTT,CEMENT
37,HONAUT,-1.07,-24.35,99.46,50.88,145087.0,-46963.0,145875.0,58192.79,-40.89,22.0,X-SC,5.68,90.0,-0.32,0.99,11.14,X40N,ATH,ELECTRICAL
82,VOLTAS,0.43,-19.24,79.78,45.20,123547.0,-36882.0,154860.0,1856.00,11.25,48.0,X-MC,21.06,79.0,-0.30,1.05,9.11,XY25,NTT,AC


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,0.13,-24.27,69.96,28.70,231829.0,-106221.0,331374.0,452.00,-31.57,54.0,X-LC,8.60,1.0,-0.46,2.26,4.29,X40,NTT,FMCG
72,TCS,0.04,-24.59,68.08,26.75,250564.0,-120031.0,368044.0,4208.34,-38.95,68.0,X-LC,5.49,2.0,-0.48,2.51,27.65,X40,ATH,IT
45,INFY,-2.62,-16.51,70.88,42.66,237207.0,-66195.0,334660.0,1972.00,-39.69,58.0,X-LC,9.43,3.0,-0.28,2.28,17.12,X40,NTT,IT
74,TMPV,2.22,-33.96,121.16,46.05,357962.0,-151926.0,295446.0,600.00,-87.07,41.0,X-LC,17.64,4.0,-0.42,2.01,3.87,XY24,NTT,AUTO
33,HCLTECH,0.38,-1.91,28.15,25.70,99672.0,-6899.0,354074.0,1837.07,-6.73,71.0,X-LC,3.21,8.0,-0.07,2.41,41.27,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,2.50,-25.89,34.92,-0.01,33076.0,-33087.0,94718.0,2282.00,738.08,43.0,M-SC,22.88,231.0,-1.00,0.65,61.22,OX40N,BTT,AC
80,VALIANTORG,4.60,-9.09,91.56,74.15,136180.0,-14872.0,148733.0,838.00,NaN,52.0,NaN,51.07,145.0,-0.11,1.01,121.34,XR,NTT,CHEMICALS
70,TANLA,3.67,-27.22,195.66,115.17,385546.0,-73712.0,197049.0,1903.93,NaN,77.0,NaN,8.15,147.0,-0.19,1.34,74.19,AR,ATH,IT
25,FINCABLES,-0.13,44.20,43.17,106.44,91909.0,65256.0,212900.0,1629.94,5.26,35.0,M-SC,26.73,220.0,0.71,1.45,61.31,OX40N,ATH,CABLES
85,ZYDUSLIFE,-0.01,31.33,3.18,35.51,8752.0,65659.0,275233.0,1285.01,29.99,71.0,H-LC,11.29,120.0,7.50,1.87,44.84,AR,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,3.67,-27.22,195.66,115.17,385546.0,-73712.0,197049.0,1903.93,NaN,77.0,NaN,8.15,147.0,-0.19,1.34,74.19,AR,ATH,IT
71,TATAELXSI,0.58,-45.06,140.32,32.03,101631.0,-59407.0,72428.0,9161.00,-37.56,66.0,H-SC,14.54,158.0,-0.58,0.49,27.24,OX40N,NTT,IT
84,WIPRO,-0.83,-9.62,113.56,93.01,256839.0,-24070.0,226170.0,420.00,-23.61,71.0,M-LC,9.01,98.0,-0.09,1.54,25.44,XR,NTT,IT
33,HCLTECH,0.38,-1.91,28.15,25.70,99672.0,-6899.0,354074.0,1837.07,-6.73,71.0,X-LC,3.21,8.0,-0.07,2.41,41.27,X40,ATH,IT
13,BSOFT,2.32,-40.08,147.99,48.59,124723.0,-56375.0,84278.0,819.60,NaN,62.0,NaN,12.24,172.0,-0.45,0.57,23.21,XR,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.15
1,25,49.28
2,50,80.16


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.52
MC    20.58
SC    20.01
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.52
XY24     22.36
X40N     18.36
XR        9.90
XY25      9.36
OX40N     6.61
AR        5.10
SR        0.78
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    39.11
X-MC    18.19
H-SC     7.97
X-SC     7.01
M-SC     5.03
H-LC     1.87
M-LC     1.54
M-MC     1.07
H-MC     1.00
L-MC     0.32
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,17.50,-33.59,81.65
IT,13.37,-30.56,98.79
FINANCE,10.45,-31.35,87.47
AUTO,8.23,-23.90,68.10
PHARMA,6.43,5.87,20.08
INSURANCE,5.84,-15.29,46.65
MISC,5.46,-53.86,115.71
PAINTS,5.25,-33.84,57.45
BANKS,4.85,-25.75,88.81


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4152513.0,19
X40,2352522.0,17
X40N,1643721.0,12
XR,1601972.0,13
AR,1043868.0,7
XY25,839203.0,6
OX40N,819191.0,10
SR,298706.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,3199771.0,19
X-MC,1738264.0,14
X-SC,1245687.0,6
H-SC,979171.0,8
M-SC,425016.0,6
H-MC,266734.0,1
M-LC,256839.0,1
M-MC,87593.0,1
L-MC,54985.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1391737.0      8
X-SC       XY24       722237.0      2
X-LC       X40N       660600.0      5
H-SC       XY24       605456.0      3
X-LC       XY24       603687.0      2
X-MC       X40N       576336.0      4
           X40        555613.0      6
X-LC       XY25       422252.0      3
X-MC       XY25       329358.0      2
X-SC       X40        283907.0      2
X-MC       XY24       276957.0      2
H-MC       XY24       266734.0      1
M-LC       XR         256839.0      1
H-SC       OX40N      251762.0      3
X-SC       X40N       239543.0      2
M-SC       OX40N      223312.0      4
           XY24       137682.0      1
H-SC       XR         121953.0      2
X-LC       XR         121495.0      1
M-MC       XY25        87593.0      1
M-SC       XR          64022.0      1
L-MC       XR          54985.0      1
H-LC       AR           8752.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 26.0 seconds
